# Electrical Fault Detection and Classification in Power Transmission Lines

---

## 1. Problem Statement

### 1.1 Business Context

The reliable and efficient operation of electrical power systems is critical for modern infrastructure, industrial operations, and economic stability. Power transmission lines, which carry electricity from generation plants to distribution networks, are susceptible to a range of electrical faults caused by environmental factors (storms, lightning, tree contact), equipment aging, insulation degradation, and human error.

When faults occur on transmission lines, they can lead to equipment damage, power outages, safety hazards, and significant financial losses. Traditional protection relay systems detect faults using predefined threshold-based rules, but these methods can be slow, limited in fault classification capability, and prone to error under unusual operating conditions.

### 1.2 Objective

This notebook builds a machine learning pipeline to detect and classify electrical faults in a three-phase power transmission line system. Using voltage and current measurements from the transmission line, the objective is to:

1. Perform binary classification to determine whether a fault exists or not (fault detection).
2. Perform multiclass classification to identify the specific type of fault (fault classification).

### 1.3 Fault Types

The dataset captures six classes of electrical conditions based on phase involvement (A, B, C) and ground connection (G):

- No Fault: Normal operating condition with no fault present.
- Line-to-Ground (LG): A single phase conductor contacts ground.
- Line-to-Line (LL): Two phase conductors come into direct contact.
- Double Line-to-Ground (LLG): Two phase conductors contact ground simultaneously.
- Three-Phase (LLL): All three phases come into contact with each other.
- Three-Phase-to-Ground (LLLG): All three phases contact ground (most severe).

### 1.4 Dataset

The dataset (classData.csv) contains simulated measurements from a three-phase power system model. Each record contains:

- G, C, B, A: Binary indicators representing which phases and ground are involved in the fault.
- Ia, Ib, Ic: Line currents for phases A, B, and C respectively.
- Va, Vb, Vc: Line voltages for phases A, B, and C respectively.

### 1.5 Approach

The notebook follows a structured machine learning workflow:

1. Data loading and initial inspection
2. Exploratory data analysis (EDA)
3. Data preprocessing and feature engineering
4. Binary classification (fault vs no-fault)
5. Multiclass classification (fault type identification)
6. Model comparison and evaluation
7. Conclusions and recommendations

## 2. Import Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries imported successfully.")

All libraries imported successfully.


## 3. Data Loading and Initial Inspection

In [5]:
# Load the dataset
df = pd.read_csv('classData.csv')

print("Dataset shape:", df.shape)
print()
print("First 10 rows:")
df.head(10)

Dataset shape: (1200, 10)

First 10 rows:


,G,C,B,A,Ia,Ib,Ic,Va,Vb,Vc
0,1,1,1,1,19.490437,-106.060248,-189.864518,81.411336,57.942417,-50.424515
1,0,1,1,1,-151.807271,140.614869,-137.354874,142.428024,102.761469,0.262982
2,0,0,0,0,-0.007738,-0.012447,-0.017787,0.074802,0.032718,-0.002779
3,0,0,1,1,55.673146,-58.717602,-0.400019,-17.947826,48.566026,-0.031762
4,0,0,0,0,-0.007184,-0.002134,0.003109,0.073768,0.042883,-0.007997
5,1,1,1,1,24.466262,-140.339799,-35.556284,60.905602,90.852479,-199.713257
6,1,0,0,1,-27.081254,0.945694,-0.231048,-35.944703,-0.236920,-0.007226
7,0,1,1,1,-83.253129,-130.897073,-48.253761,123.250268,34.604352,-26.416035
8,0,0,1,1,-33.688225,29.395119,-0.041719,-37.034599,-34.006134,-0.724823
9,0,1,1,1,81.312022,-38.989458,168.016021,45.076521,-122.874307,134.356970


In [6]:
# Check data types and non-null counts
print("Data Types and Info:")
print()
df.info()

Data Types and Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   G       1200 non-null   int64  
 1   C       1200 non-null   int64  
 2   B       1200 non-null   int64  
 3   A       1200 non-null   int64  
 4   Ia      1200 non-null   float64
 5   Ib      1200 non-null   float64
 6   Ic      1200 non-null   float64
 7   Va      1200 non-null   float64
 8   Vb      1200 non-null   float64
 9   Vc      1200 non-null   float64
dtypes: float64(6), int64(4)
memory usage: 93.9 KB


In [7]:
# Statistical summary
print("Statistical Summary:")
df.describe()

Statistical Summary:


,G,C,B,A,Ia,Ib,Ic,Va,Vb,Vc
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,0.500000,0.333333,0.666667,0.833333,-3.001848,-1.781316,-1.718016,-1.812843,0.565530,-2.499709
std,0.500208,0.471601,0.471601,0.372833,88.280172,83.530646,72.276579,69.244418,66.225293,60.582875
min,0.000000,0.000000,0.000000,0.000000,-246.605913,-249.799228,-249.605244,-192.656646,-199.061010,-199.902826
25%,0.000000,0.000000,0.000000,1.000000,-56.042546,-36.499684,-0.536878,-39.878981,-30.437268,-0.501046
50%,0.500000,0.000000,1.000000,1.000000,-0.000156,-0.002654,-0.001134,-0.011509,0.002899,-0.003872
75%,1.000000,1.000000,1.000000,1.000000,51.473264,30.484308,0.445206,34.406369,27.878408,0.409252
max,1.000000,1.000000,1.000000,1.000000,248.711470,249.452350,235.609956,198.320947,196.674456,195.739806


In [8]:
# Check for missing values
print("Missing Values per Column:")
print(df.isnull().sum())
print()
print("Total missing values:", df.isnull().sum().sum())

Missing Values per Column:
G     0
C     0
B     0
A     0
Ia    0
Ib    0
Ic    0
Va    0
Vb    0
Vc    0
dtype: int64

Total missing values: 0


In [9]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

Number of duplicate rows: 0


## 4. Target Variable Engineering

Before performing EDA, we create the target variables needed for both binary and multiclass classification tasks. The fault type is determined by the combination of the four binary indicator columns (G, C, B, A).

In [11]:
# Create a fault type label based on G, C, B, A combinations
def get_fault_type(row):
    g, c, b, a = int(row['G']), int(row['C']), int(row['B']), int(row['A'])
    if g == 0 and c == 0 and b == 0 and a == 0:
        return 'No Fault'
    elif g == 1 and c == 0 and b == 0 and a == 1:
        return 'LG Fault'
    elif g == 0 and c == 0 and b == 1 and a == 1:
        return 'LL Fault'
    elif g == 1 and c == 0 and b == 1 and a == 1:
        return 'LLG Fault'
    elif g == 0 and c == 1 and b == 1 and a == 1:
        return 'LLL Fault'
    elif g == 1 and c == 1 and b == 1 and a == 1:
        return 'LLLG Fault'
    else:
        return 'Other'

df['Fault_Type'] = df.apply(get_fault_type, axis=1)

# Binary target: 0 = No Fault, 1 = Fault
df['Fault_Binary'] = (df['Fault_Type'] != 'No Fault').astype(int)

print("Fault Type Distribution:")
print(df['Fault_Type'].value_counts())
print()
print("Binary Fault Distribution:")
print(df['Fault_Binary'].value_counts())

Fault Type Distribution:
Fault_Type
LLLG Fault    200
LLL Fault     200
No Fault      200
LL Fault      200
LG Fault      200
LLG Fault     200
Name: count, dtype: int64

Binary Fault Distribution:
Fault_Binary
1    1000
0     200
Name: count, dtype: int64


## 5. Exploratory Data Analysis (EDA)

This section examines the distribution patterns of electrical measurements across different fault conditions to understand how voltage and current signals differ between normal operation and various fault types.

### 5.1 Fault Type Distribution

In [14]:
# Distribution of fault types
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Multiclass distribution
fault_counts = df['Fault_Type'].value_counts()
axes[0].bar(fault_counts.index, fault_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Fault Types', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fault Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(fault_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Binary distribution
binary_counts = df['Fault_Binary'].value_counts()
labels = ['No Fault', 'Fault']
colors = ['#2ecc71', '#e74c3c']
axes[1].bar(labels, binary_counts.values, color=colors, edgecolor='black')
axes[1].set_title('Binary Fault Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Condition')
axes[1].set_ylabel('Count')
for i, v in enumerate(binary_counts.values):
    axes[1].text(i, v + 3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.2 Distribution of Electrical Measurements

In [16]:
# Distribution of current and voltage features
measurement_cols = ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(measurement_cols):
    axes[idx].hist(df[col], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title('Distribution of ' + col, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5,
                      label='Mean: {:.2f}'.format(df[col].mean()))
    axes[idx].legend(fontsize=9)

plt.suptitle('Distribution of Current and Voltage Measurements',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Box Plots by Fault Type

In [18]:
# Box plots of measurements by fault type
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(measurement_cols):
    data_groups = []
    labels_list = sorted(df['Fault_Type'].unique())
    for ft in labels_list:
        data_groups.append(df[df['Fault_Type'] == ft][col].values)
    bp = axes[idx].boxplot(data_groups, labels=labels_list, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('steelblue')
        patch.set_alpha(0.6)
    axes[idx].set_title(col + ' by Fault Type', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Fault Type')
    axes[idx].set_ylabel(col)
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Electrical Measurements by Fault Type',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.4 Correlation Analysis

In [20]:
# Correlation heatmap for numerical features
fig, ax = plt.subplots(figsize=(10, 8))
corr_cols = ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc', 'G', 'C', 'B', 'A']
corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix of Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.5 Pairwise Scatter Plots for Current Features

In [22]:
# Scatter plots of current features colored by fault type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [('Ia', 'Ib'), ('Ib', 'Ic'), ('Ia', 'Ic')]
fault_types_sorted = sorted(df['Fault_Type'].unique())
colors_map = {}
palette = ['green', 'blue', 'orange', 'red', 'purple', 'brown']
for i, ft in enumerate(fault_types_sorted):
    colors_map[ft] = palette[i]

for idx, (x_col, y_col) in enumerate(pairs):
    for ft in fault_types_sorted:
        subset = df[df['Fault_Type'] == ft]
        axes[idx].scatter(subset[x_col], subset[y_col], label=ft, alpha=0.5,
                         s=15, color=colors_map[ft])
    axes[idx].set_xlabel(x_col, fontsize=11)
    axes[idx].set_ylabel(y_col, fontsize=11)
    axes[idx].set_title(x_col + ' vs ' + y_col, fontsize=12, fontweight='bold')

axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.suptitle('Current Relationships by Fault Type',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.6 Voltage Feature Scatter Plots

In [24]:
# Scatter plots of voltage features colored by fault type
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs_v = [('Va', 'Vb'), ('Vb', 'Vc'), ('Va', 'Vc')]

for idx, (x_col, y_col) in enumerate(pairs_v):
    for ft in fault_types_sorted:
        subset = df[df['Fault_Type'] == ft]
        axes[idx].scatter(subset[x_col], subset[y_col], label=ft, alpha=0.5,
                         s=15, color=colors_map[ft])
    axes[idx].set_xlabel(x_col, fontsize=11)
    axes[idx].set_ylabel(y_col, fontsize=11)
    axes[idx].set_title(x_col + ' vs ' + y_col, fontsize=12, fontweight='bold')

axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.suptitle('Voltage Relationships by Fault Type',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.7 Statistical Summary by Fault Type

In [26]:
# Mean of each measurement grouped by fault type
print("Mean Values of Electrical Measurements by Fault Type:")
print()
summary = df.groupby('Fault_Type')[measurement_cols].agg(['mean', 'std']).round(4)
print(summary)

Mean Values of Electrical Measurements by Fault Type:

                 Ia                 Ib                 Ic                Va  \
               mean       std     mean       std     mean       std    mean   
Fault_Type                                                                    
LG Fault    -0.0562   57.5841  -0.0004    0.5056   0.0058    0.5083 -2.7395   
LL Fault     1.0683   56.3570  -0.8810   56.2964  -0.0663    0.4916 -1.7461   
LLG Fault    6.3409   82.4988   3.9117   86.2895   0.0054    0.4986 -2.4606   
LLL Fault  -14.9690  115.9877  -2.5350  114.9236   6.2554  111.1641 -4.7210   
LLLG Fault -10.3966  141.0101 -11.1827  134.4898 -16.5085  137.1898  0.7906   
No Fault     0.0015    0.0088  -0.0005    0.0098   0.0001    0.0099 -0.0005   

                          Vb                 Vc            
                 std    mean       std     mean       std  
Fault_Type                                                 
LG Fault     28.7385 -0.0321    0.4921   0.0398    0.

## 6. Data Preprocessing

This section prepares the data for modeling by defining features, scaling measurements, and splitting data into training and test sets for both binary and multiclass classification tasks.

### 6.1 Define Features and Targets

In [29]:
# Feature columns: use the electrical measurements
feature_cols = ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc']

X = df[feature_cols].copy()

# Binary target
y_binary = df['Fault_Binary'].copy()

# Multiclass target (encode fault types as integers)
le = LabelEncoder()
y_multi = le.fit_transform(df['Fault_Type'])
class_names = le.classes_

print("Feature columns:", feature_cols)
print("Number of features:", len(feature_cols))
print()
print("Binary target classes:", sorted(y_binary.unique()))
print("Multiclass target classes:", list(class_names))
print("Multiclass encoded values:", sorted(set(y_multi)))

Feature columns: ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc']
Number of features: 6

Binary target classes: [0, 1]
Multiclass target classes: ['LG Fault', 'LL Fault', 'LLG Fault', 'LLL Fault', 'LLLG Fault', 'No Fault']
Multiclass encoded values: [0, 1, 2, 3, 4, 5]


### 6.2 Feature Scaling

In [31]:
# Standardize the features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print("Scaled feature statistics:")
print(X_scaled.describe().round(4))

Scaled feature statistics:
              Ia         Ib         Ic         Va         Vb         Vc
count  1200.0000  1200.0000  1200.0000  1200.0000  1200.0000  1200.0000
mean     -0.0000     0.0000    -0.0000    -0.0000    -0.0000    -0.0000
std       1.0004     1.0004     1.0004     1.0004     1.0004     1.0004
min      -2.7606    -2.9704    -3.4311    -2.7572    -3.0156    -3.2598
25%      -0.6011    -0.4158     0.0163    -0.5500    -0.4683     0.0330
50%       0.0340     0.0213     0.0238     0.0260    -0.0085     0.0412
75%       0.6173     0.3864     0.0299     0.5233     0.4126     0.0480
max       2.8525     3.0089     3.2850     2.8915     2.9625     3.2736


### 6.3 Train-Test Split

In [33]:
# Split for binary classification
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_scaled, y_binary, test_size=0.2, random_state=42, stratify=y_binary)

# Split for multiclass classification
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_scaled, y_multi, test_size=0.2, random_state=42, stratify=y_multi)

print("Binary Classification Split:")
print("  Training set: {} samples".format(X_train_bin.shape[0]))
print("  Test set:     {} samples".format(X_test_bin.shape[0]))
print()
print("Multiclass Classification Split:")
print("  Training set: {} samples".format(X_train_multi.shape[0]))
print("  Test set:     {} samples".format(X_test_multi.shape[0]))

Binary Classification Split:
  Training set: 960 samples
  Test set:     240 samples

Multiclass Classification Split:
  Training set: 960 samples
  Test set:     240 samples


## 7. Binary Classification: Fault Detection

In this section, multiple machine learning models are trained to detect whether a fault exists in the transmission line (binary: fault or no fault).

### 7.1 Model Training and Evaluation

In [36]:
# Define models for binary classification
binary_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, use_label_encoder=False,
                              eval_metric='logloss', random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

binary_results = {}

print("Binary Classification Results")
print("=" * 70)

for name, model in binary_models.items():
    # Train
    model.fit(X_train_bin, y_train_bin)

    # Predict
    y_pred = model.predict(X_test_bin)

    # Evaluate
    acc = accuracy_score(y_test_bin, y_pred)
    prec = precision_score(y_test_bin, y_pred)
    rec = recall_score(y_test_bin, y_pred)
    f1 = f1_score(y_test_bin, y_pred)

    binary_results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'Predictions': y_pred
    }

    print()
    print("{}:".format(name))
    print("  Accuracy:  {:.4f}".format(acc))
    print("  Precision: {:.4f}".format(prec))
    print("  Recall:    {:.4f}".format(rec))
    print("  F1 Score:  {:.4f}".format(f1))

Binary Classification Results

Logistic Regression:
  Accuracy:  0.8333
  Precision: 0.8333
  Recall:    1.0000
  F1 Score:  0.9091

Decision Tree:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

Random Forest:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

XGBoost:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000

SVM:
  Accuracy:  0.9500
  Precision: 1.0000
  Recall:    0.9400
  F1 Score:  0.9691

KNN:
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000


### 7.2 Binary Classification - Confusion Matrices

In [38]:
# Plot confusion matrices for binary classification
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, result) in enumerate(binary_results.items()):
    cm = confusion_matrix(y_test_bin, result['Predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['No Fault', 'Fault'],
                yticklabels=['No Fault', 'Fault'])
    axes[idx].set_title(name + ' (Acc: {:.4f})'.format(result['Accuracy']),
                       fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices - Binary Classification (Fault Detection)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 7.3 Binary Classification - Model Comparison

In [40]:
# Compare binary classification models
comparison_bin = pd.DataFrame({
    name: {k: v for k, v in result.items() if k != 'Predictions'}
    for name, result in binary_results.items()
}).T

print("Binary Classification Model Comparison:")
print(comparison_bin.round(4))

fig, ax = plt.subplots(figsize=(12, 6))
comparison_bin.plot(kind='bar', ax=ax, edgecolor='black', width=0.7)
ax.set_title('Binary Classification - Model Performance Comparison',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

Binary Classification Model Comparison:
                     Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.8333     0.8333    1.00    0.9091
Decision Tree          1.0000     1.0000    1.00    1.0000
Random Forest          1.0000     1.0000    1.00    1.0000
XGBoost                1.0000     1.0000    1.00    1.0000
SVM                    0.9500     1.0000    0.94    0.9691
KNN                    1.0000     1.0000    1.00    1.0000


### 7.4 Cross-Validation for Binary Classification

In [42]:
# Cross-validation for binary classification models
print("5-Fold Cross-Validation Scores (Binary Classification)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, use_label_encoder=False,
                              eval_metric='logloss', random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

cv_results_bin = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_scaled, y_binary, cv=cv, scoring='accuracy')
    cv_results_bin[name] = scores
    print("{}:".format(name))
    print("  Mean Accuracy: {:.4f} (+/- {:.4f})".format(scores.mean(), scores.std()))
    print("  Fold scores:   {}".format([round(s, 4) for s in scores]))
    print()

5-Fold Cross-Validation Scores (Binary Classification)
Logistic Regression:
  Mean Accuracy: 0.8333 (+/- 0.0000)
  Fold scores:   [0.8333, 0.8333, 0.8333, 0.8333, 0.8333]

Decision Tree:
  Mean Accuracy: 0.9992 (+/- 0.0017)
  Fold scores:   [1.0, 1.0, 0.9958, 1.0, 1.0]

Random Forest:
  Mean Accuracy: 1.0000 (+/- 0.0000)
  Fold scores:   [1.0, 1.0, 1.0, 1.0, 1.0]

XGBoost:
  Mean Accuracy: 0.9992 (+/- 0.0017)
  Fold scores:   [1.0, 1.0, 0.9958, 1.0, 1.0]

SVM:
  Mean Accuracy: 0.9375 (+/- 0.0070)
  Fold scores:   [0.9417, 0.9375, 0.9375, 0.9458, 0.925]

KNN:
  Mean Accuracy: 0.9975 (+/- 0.0033)
  Fold scores:   [0.9958, 1.0, 1.0, 1.0, 0.9917]



## 8. Multiclass Classification: Fault Type Identification

This section trains and evaluates the same set of machine learning models for classifying the specific type of electrical fault.

### 8.1 Model Training and Evaluation

In [45]:
# Define models for multiclass classification
multi_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, use_label_encoder=False,
                              eval_metric='mlogloss', random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

multi_results = {}

print("Multiclass Classification Results")
print("=" * 70)

for name, model in multi_models.items():
    # Train
    model.fit(X_train_multi, y_train_multi)

    # Predict
    y_pred = model.predict(X_test_multi)

    # Evaluate
    acc = accuracy_score(y_test_multi, y_pred)
    prec = precision_score(y_test_multi, y_pred, average='weighted')
    rec = recall_score(y_test_multi, y_pred, average='weighted')
    f1 = f1_score(y_test_multi, y_pred, average='weighted')

    multi_results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'Predictions': y_pred
    }

    print()
    print("{}:".format(name))
    print("  Accuracy:  {:.4f}".format(acc))
    print("  Precision: {:.4f}".format(prec))
    print("  Recall:    {:.4f}".format(rec))
    print("  F1 Score:  {:.4f}".format(f1))

Multiclass Classification Results

Logistic Regression:
  Accuracy:  0.3875
  Precision: 0.3712
  Recall:    0.3875
  F1 Score:  0.3540

Decision Tree:
  Accuracy:  0.8417
  Precision: 0.8466
  Recall:    0.8417
  F1 Score:  0.8413

Random Forest:
  Accuracy:  0.9000
  Precision: 0.9011
  Recall:    0.9000
  F1 Score:  0.8999

XGBoost:
  Accuracy:  0.8708
  Precision: 0.8722
  Recall:    0.8708
  F1 Score:  0.8706

SVM:
  Accuracy:  0.7292
  Precision: 0.7422
  Recall:    0.7292
  F1 Score:  0.7256

KNN:
  Accuracy:  0.7000
  Precision: 0.7109
  Recall:    0.7000
  F1 Score:  0.6859


### 8.2 Detailed Classification Reports

In [47]:
# Print detailed classification reports for top performing models
top_models = ['Random Forest', 'XGBoost', 'Decision Tree']

for name in top_models:
    print("Classification Report - {}".format(name))
    print("=" * 60)
    print(classification_report(y_test_multi, multi_results[name]['Predictions'],
                                target_names=class_names))
    print()

Classification Report - Random Forest
              precision    recall  f1-score   support

    LG Fault       1.00      1.00      1.00        40
    LL Fault       0.88      0.88      0.88        40
   LLG Fault       0.88      0.88      0.88        40
   LLL Fault       0.80      0.88      0.83        40
  LLLG Fault       0.86      0.78      0.82        40
    No Fault       1.00      1.00      1.00        40

    accuracy                           0.90       240
   macro avg       0.90      0.90      0.90       240
weighted avg       0.90      0.90      0.90       240


Classification Report - XGBoost
              precision    recall  f1-score   support

    LG Fault       1.00      0.97      0.99        40
    LL Fault       0.82      0.93      0.87        40
   LLG Fault       0.86      0.78      0.82        40
   LLL Fault       0.78      0.78      0.78        40
  LLLG Fault       0.78      0.78      0.78        40
    No Fault       1.00      1.00      1.00        40

    ac

### 8.3 Multiclass Classification - Confusion Matrices

In [49]:
# Plot confusion matrices for multiclass classification
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, (name, result) in enumerate(multi_results.items()):
    cm = confusion_matrix(y_test_multi, result['Predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=class_names, yticklabels=class_names)
    axes[idx].set_title(name + ' (Acc: {:.4f})'.format(result['Accuracy']),
                       fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].tick_params(axis='y', rotation=0)

plt.suptitle('Confusion Matrices - Multiclass Classification (Fault Type)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 8.4 Multiclass Classification - Model Comparison

In [51]:
# Compare multiclass classification models
comparison_multi = pd.DataFrame({
    name: {k: v for k, v in result.items() if k != 'Predictions'}
    for name, result in multi_results.items()
}).T

print("Multiclass Classification Model Comparison:")
print(comparison_multi.round(4))

fig, ax = plt.subplots(figsize=(12, 6))
comparison_multi.plot(kind='bar', ax=ax, edgecolor='black', width=0.7)
ax.set_title('Multiclass Classification - Model Performance Comparison',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

Multiclass Classification Model Comparison:
                     Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.3875     0.3712  0.3875    0.3540
Decision Tree          0.8417     0.8466  0.8417    0.8413
Random Forest          0.9000     0.9011  0.9000    0.8999
XGBoost                0.8708     0.8722  0.8708    0.8706
SVM                    0.7292     0.7422  0.7292    0.7256
KNN                    0.7000     0.7109  0.7000    0.6859


### 8.5 Cross-Validation for Multiclass Classification

In [53]:
# Cross-validation for multiclass classification
print("5-Fold Cross-Validation Scores (Multiclass Classification)")
print("=" * 60)

cv_models_multi = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, use_label_encoder=False,
                              eval_metric='mlogloss', random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

cv_results_multi = {}
for name, model in cv_models_multi.items():
    scores = cross_val_score(model, X_scaled, y_multi, cv=cv, scoring='accuracy')
    cv_results_multi[name] = scores
    print("{}:".format(name))
    print("  Mean Accuracy: {:.4f} (+/- {:.4f})".format(scores.mean(), scores.std()))
    print("  Fold scores:   {}".format([round(s, 4) for s in scores]))
    print()

5-Fold Cross-Validation Scores (Multiclass Classification)
Logistic Regression:
  Mean Accuracy: 0.3617 (+/- 0.0165)
  Fold scores:   [0.35, 0.3792, 0.3417, 0.3542, 0.3833]

Decision Tree:
  Mean Accuracy: 0.8617 (+/- 0.0135)
  Fold scores:   [0.8625, 0.8625, 0.8375, 0.8792, 0.8667]

Random Forest:
  Mean Accuracy: 0.8992 (+/- 0.0200)
  Fold scores:   [0.8875, 0.9375, 0.8875, 0.9, 0.8833]

XGBoost:
  Mean Accuracy: 0.8842 (+/- 0.0180)
  Fold scores:   [0.8625, 0.9125, 0.8792, 0.8958, 0.8708]

SVM:
  Mean Accuracy: 0.7108 (+/- 0.0207)
  Fold scores:   [0.725, 0.7417, 0.6958, 0.7083, 0.6833]

KNN:
  Mean Accuracy: 0.7125 (+/- 0.0173)
  Fold scores:   [0.7042, 0.7375, 0.7083, 0.6875, 0.725]



## 9. Feature Importance Analysis

Understanding which electrical measurements contribute most to fault detection helps domain experts validate the model behavior and provides insight into the physical characteristics of different fault types.

In [55]:
# Feature importance from Random Forest (multiclass)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_multi, y_train_multi)

importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['Feature'], importance_df['Importance'],
        color='steelblue', edgecolor='black')
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Random Forest Feature Importance (Multiclass)',
             fontsize=13, fontweight='bold')

for i, (val, name) in enumerate(zip(importance_df['Importance'], importance_df['Feature'])):
    ax.text(val + 0.005, i, '{:.4f}'.format(val), va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("Feature Importance Rankings:")
print(importance_df.sort_values('Importance', ascending=False).to_string(index=False))

Feature Importance Rankings:
Feature  Importance
     Ib    0.217236
     Ia    0.169126
     Ic    0.168472
     Vb    0.164276
     Vc    0.161858
     Va    0.119031


In [56]:
# Feature importance from XGBoost (multiclass)
xgb_model = XGBClassifier(n_estimators=100, use_label_encoder=False,
                           eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train_multi, y_train_multi)

xgb_importances = xgb_model.feature_importances_
xgb_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(xgb_importance_df['Feature'], xgb_importance_df['Importance'],
        color='coral', edgecolor='black')
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('XGBoost Feature Importance (Multiclass)',
             fontsize=13, fontweight='bold')

for i, (val, name) in enumerate(zip(xgb_importance_df['Importance'], xgb_importance_df['Feature'])):
    ax.text(val + 0.005, i, '{:.4f}'.format(val), va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 10. Overall Model Comparison

This section provides a consolidated view comparing all models across both classification tasks to identify the best performing algorithms for electrical fault detection and classification.

In [58]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Binary comparison
models_list = list(binary_results.keys())
bin_accs = [binary_results[m]['Accuracy'] for m in models_list]
axes[0].bar(models_list, bin_accs, color='steelblue', edgecolor='black')
axes[0].set_title('Binary Classification Accuracy', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.8, 1.02)
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(bin_accs):
    axes[0].text(i, v + 0.003, '{:.4f}'.format(v), ha='center', fontsize=9, fontweight='bold')

# Multiclass comparison
multi_accs = [multi_results[m]['Accuracy'] for m in models_list]
axes[1].bar(models_list, multi_accs, color='coral', edgecolor='black')
axes[1].set_title('Multiclass Classification Accuracy', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.5, 1.05)
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(multi_accs):
    axes[1].text(i, v + 0.008, '{:.4f}'.format(v), ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Comparison Across Tasks',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [59]:
# Final summary table
print("=" * 80)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 80)
print()

summary_data = []
for name in models_list:
    summary_data.append({
        'Model': name,
        'Binary Accuracy': binary_results[name]['Accuracy'],
        'Binary F1': binary_results[name]['F1 Score'],
        'Multiclass Accuracy': multi_results[name]['Accuracy'],
        'Multiclass F1': multi_results[name]['F1 Score']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.set_index('Model')
print(summary_df.round(4))
print()

# Best models
best_binary = summary_df['Binary Accuracy'].idxmax()
best_multi = summary_df['Multiclass Accuracy'].idxmax()
print("Best model for Binary Classification:     {} (Accuracy: {:.4f})".format(
    best_binary, summary_df.loc[best_binary, 'Binary Accuracy']))
print("Best model for Multiclass Classification: {} (Accuracy: {:.4f})".format(
    best_multi, summary_df.loc[best_multi, 'Multiclass Accuracy']))

FINAL MODEL PERFORMANCE SUMMARY

                     Binary Accuracy  Binary F1  Multiclass Accuracy  \
Model                                                                  
Logistic Regression           0.8333     0.9091               0.3875   
Decision Tree                 1.0000     1.0000               0.8417   
Random Forest                 1.0000     1.0000               0.9000   
XGBoost                       1.0000     1.0000               0.8708   
SVM                           0.9500     0.9691               0.7292   
KNN                           1.0000     1.0000               0.7000   

                     Multiclass F1  
Model                               
Logistic Regression         0.3540  
Decision Tree               0.8413  
Random Forest               0.8999  
XGBoost                     0.8706  
SVM                         0.7256  
KNN                         0.6859  

Best model for Binary Classification:     Decision Tree (Accuracy: 1.0000)
Best model for Mul

## 11. Conclusions and Recommendations

### Key Findings

1. Data Characteristics: The dataset contains 1200 records of voltage and current measurements across six fault categories. The data is clean with no missing values or duplicates, and is balanced across fault classes.

2. Binary Classification (Fault Detection): All tree-based models (Decision Tree, Random Forest, XGBoost) and SVM achieved near-perfect accuracy in detecting whether a fault exists. Logistic Regression showed comparatively lower performance due to the non-linear nature of fault signatures in electrical signals.

3. Multiclass Classification (Fault Type Identification): Random Forest, Decision Tree, and XGBoost classifiers demonstrated strong performance in identifying specific fault types. The models can effectively distinguish between different fault configurations based on current and voltage patterns.

4. Feature Importance: Current measurements (Ia, Ib, Ic) and voltage measurements (Va, Vb, Vc) both contribute significantly to fault classification. The relative importance varies by fault type, reflecting the physical behavior of different fault conditions.

### Recommendations

- For deployment in real-time fault detection systems, Random Forest or XGBoost are recommended due to their high accuracy and robustness.
- The pipeline should be validated on real-world operational data from actual transmission lines before production deployment.
- Periodic model retraining is recommended to account for changes in system characteristics over time.
- Additional features such as power factor, impedance, and frequency domain measurements could further improve classification performance.

### Limitations

- The dataset is based on simulated data, which may not capture all real-world variations and noise.
- The model does not account for transient fault behavior or time-series dynamics of fault signals.
- High-impedance faults and intermittent faults may require specialized detection approaches not covered here.

---

End of Notebook